# 02 -- Processing

Cleans, aligns, and feature-engineers ALL data sources.

**Inputs:** `data/raw/` | **Outputs:** `data/processed/`

Fixes vs. original: no weekend interpolation, publication lags, forward-fill only, one notebook instead of 10.

In [1]:
import pandas as pd
import numpy as np
import os, sys

PROJECT_DIR = os.path.dirname(os.path.abspath('__file__'))
RAW_DIR = os.path.join(PROJECT_DIR, 'data', 'raw')
PROCESSED_DIR = os.path.join(PROJECT_DIR, 'data', 'processed')
os.makedirs(PROCESSED_DIR, exist_ok=True)

sys.path.insert(0, os.path.join(PROJECT_DIR, 'shared'))
from macro_utils import *

## 1. S&P 500 Index

In [2]:
sp500 = pd.read_csv(os.path.join(RAW_DIR, 'sp500', 'sp500_index.csv'), parse_dates=['caldt'])
sp500 = sp500.set_index('caldt')
sp500.index.name = 'date'

# Keep only trading days! No weekend interpolation.
sp500 = sp500.ffill()

# Features
sp500 = add_return_features(sp500, 'spindx', 'sprtrn')
sp500 = add_pico_features(sp500, 'spindx', threshold=0.04)
sp500 = add_rolling_stats(sp500, 'spindx')

sp500.to_csv(os.path.join(PROCESSED_DIR, 'sp500_index.csv'))
print(f"sp500_index.csv: {sp500.shape}")

sp500_index.csv: (8817, 38)


## 2. Macro Data (FRED)

In [3]:
MACROS_DIR = os.path.join(RAW_DIR, 'macros')

publication_lags = {
    'gdp': 90, 'gdp_per_capita': 90, 'gdp_monthly': 30,
    'gdp_euro': 90, 'gdp_china': 90,
    'corporate_earnings': 60, 'gov_spending': 60, 'gov_deficit': 60,
    'gov_debt': 45, 'wealth_inequality': 90,
    'unemployment': 5, 'retail_sales': 15,
    'inflation': 15, 'cpi': 15, 'cpi_median': 15,
    'consumer_sentiment': 1, 'money_supply_m1': 20,
    'money_velocity': 60, 'electricity': 15, 'ppi': 15,
}

if os.path.exists(MACROS_DIR):
    macros_d, macros_m, macros_q, macros_y = process_macro_folder(
        MACROS_DIR, publication_lags=publication_lags, verbose=True)
    for name, df in [('macros_daily', macros_d), ('macros_monthly', macros_m), ('macros_quarterly', macros_q)]:
        if not df.empty:
            df.to_csv(os.path.join(PROCESSED_DIR, f'{name}.csv'))
            print(f"{name}.csv: {df.shape}")
else:
    print("No macros directory found.")

  bonds_yahoo: daily, 2000-09-21 → 2025-06-03, 6207 rows
  commodities: monthly, 1913-01-01 → 2025-04-01, 1348 rows
  commodities_custom: daily, 2000-09-15 → 2025-05-12, 9006 rows
  consumer_sentiment: monthly, 1952-11-02 → 2025-03-02, 869 rows
  corporate_quarter_financial_reports: quarterly, 2009-10-01 → 2024-10-01, 61 rows
  cpi: monthly, 1955-02-16 → 2024-03-16, 830 rows
  cpi_euro: monthly, 1991-01-01 → 2023-01-01, 385 rows
  deuda: quarterly, 1966-01-01 → 2024-10-01, 236 rows
  electricity: monthly, 1978-11-16 → 2025-04-16, 558 rows
  euro_dolar: daily, 1999-01-04 → 2025-05-16, 6880 rows
  federal_surplus_deficit: quarterly, 1959-07-01 → 2023-10-01, 258 rows
  gdp: monthly, 1960-03-31 → 2025-05-30, 783 rows
  gdp_asia: monthly, 1979-01-01 → 2023-10-01, 538 rows
  gdp_china: monthly, 1978-04-01 → 2024-02-29, 552 rows
  gdp_euro: monthly, 1961-05-30 → 2022-10-30, 738 rows
  gdp_per_capita: quarterly, 1947-04-01 → 2025-04-01, 313 rows
  government_bonds: quarterly, 1953-04-01 → 2025

## 3. Commodities

In [4]:
COMMODITIES_DIR = os.path.join(RAW_DIR, 'commodities')

if os.path.exists(COMMODITIES_DIR):
    comm_d, _, _, _ = process_macro_folder(COMMODITIES_DIR, verbose=True)
    comm_d = trim_nan_edges(comm_d).ffill()
    comm_d.to_csv(os.path.join(PROCESSED_DIR, 'commodities_daily.csv'))
    print(f"commodities_daily.csv: {comm_d.shape}")
else:
    print("No commodities directory found.")

  coffee: daily, 2000-01-03 → 2025-06-03, 6371 rows
  copper: daily, 2000-08-30 → 2025-06-03, 6216 rows
  corn: daily, 2000-07-17 → 2025-06-03, 6224 rows
  cotton: daily, 2000-01-03 → 2025-06-03, 6373 rows
  gold: daily, 2000-08-30 → 2025-06-03, 6211 rows
  natural_gas: daily, 1997-01-07 → 2025-05-27, 7406 rows
  oil: daily, 1986-01-02 → 2025-05-12, 10268 rows
  platinum: daily, 1997-10-29 → 2025-06-03, 6239 rows
  silver: daily, 2000-08-30 → 2025-06-03, 6213 rows
  soybeans: daily, 2000-09-15 → 2025-06-03, 6216 rows
  sugar: daily, 2000-03-01 → 2025-06-03, 6334 rows
  utilities: daily, 1998-12-22 → 2025-06-03, 6652 rows
  wheat: daily, 2000-07-17 → 2025-06-03, 6236 rows

Processed 13 series.
commodities_daily.csv: (6167, 78)


## 4. S&P 500 Constituents

In [5]:
CONST_PATH = os.path.join(RAW_DIR, 'constituents', 'sp500_constituents.parquet')

if os.path.exists(CONST_PATH):
    print("Loading constituents...")
    stocks = pd.read_parquet(CONST_PATH)
    stocks['DlyCalDt'] = pd.to_datetime(stocks['DlyCalDt'])
    print(f"Loaded {len(stocks)} rows. Computing daily stats...")
    cstats = compute_daily_constituent_stats(stocks, date_col='DlyCalDt')
    cstats.to_csv(os.path.join(PROCESSED_DIR, 'constituent_stats.csv'))
    print(f"constituent_stats.csv: {cstats.shape}")
else:
    print("No constituents parquet found.")

Loading constituents...
Loaded 3038134 rows. Computing daily stats...
constituent_stats.csv: (6289, 33)


## 5. Financial Ratios

In [6]:
RATIOS_PATH = os.path.join(RAW_DIR, 'financial_ratios.parquet')

# Publication lag for financial ratios:
# - Prefer 'public_date' column if present (WRDS already lags by 2mo to avoid lookahead)
# - Else fall back to 'adate' (period-end) and apply a 60-day shift to match WRDS convention
# Reference: WRDS Financial Ratios Suite documentation explicitly lags observations
# by two months to ensure all data is publicly available at each timestamp.
RATIOS_PUB_LAG_DAYS = 60

if os.path.exists(RATIOS_PATH):
    ratios = pd.read_parquet(RATIOS_PATH)

    if 'public_date' in ratios.columns:
        # Preferred path: public_date is already publication-safe
        ratios['public_date'] = pd.to_datetime(ratios['public_date'])
        sp_ratios = (ratios
                     .drop(columns=['permno', 'adate', 'qdate'], errors='ignore')
                     .groupby('public_date').mean(numeric_only=True)
                     .sort_index())
        sp_ratios.index.name = 'date'
        print(f"[ratios] Used public_date (WRDS pre-lagged) — no shift needed")
    else:
        # Fallback: apply 60-day lag to adate
        ratios['adate'] = pd.to_datetime(ratios['adate'])
        sp_ratios = (ratios
                     .drop(columns=['permno'], errors='ignore')
                     .groupby('adate').mean(numeric_only=True)
                     .sort_index())
        sp_ratios.index = sp_ratios.index + pd.Timedelta(days=RATIOS_PUB_LAG_DAYS)
        sp_ratios.index.name = 'date'
        print(f"[ratios] No public_date column — applied {RATIOS_PUB_LAG_DAYS}d lag to adate")

    sp_ratios.to_csv(os.path.join(PROCESSED_DIR, 'sp_ratios_monthly.csv'))
    print(f"sp_ratios_monthly.csv: {sp_ratios.shape}")
else:
    print("No ratios parquet found.")

print("\n=== Processing complete ===")

[ratios] Used public_date (WRDS pre-lagged) — no shift needed
sp_ratios_monthly.csv: (660, 69)

=== Processing complete ===
